# Sử dụng Tensorflow để optimize & deploy model

  ‎ 

# Convert model   

In [12]:
import os
import tensorflow as tf 
import tf_keras
import numpy  as np
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import models
import onnx
import onnxscript
import onnxruntime as ort

In [13]:
import onnx_graphsurgeon as gs
import onnxruntime as ort
import sng4onnx
import sne4onnx
from onnx2tf import convert

In [14]:
NUM_CLASSES = 10

model = models.mobilenet_v3_small(weights=None)

model.classifier[-1] = nn.Linear(
    model.classifier[-1].in_features,
    NUM_CLASSES
)

model.load_state_dict(
    torch.load(
        "garbage_classifier_best.pth",
        map_location="cpu"
    )
)

model.eval()

dummy_input = torch.randn(1, 3, 256, 256)

In [15]:
onnx_program = torch.onnx.export(
    model,
    (dummy_input,),
    input_names=["input"],
    output_names=["output"],
    dynamo=True
)

onnx_program.save("garbage_classifier.onnx")

[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


‎ ‎ 

# Tensorflow

In [16]:
import importlib

converter_module = importlib.import_module("onnx2tf.onnx2tf")

def sample_test_data():
    return np.random.default_rng(42).random(
        (20, 256, 256, 3),
        dtype=np.float32
    )

converter_module.download_test_image_data = sample_test_data

converter_module.convert(
    input_onnx_file_path="garbage_classifier.onnx",
    output_folder_path="converted_model_retry",
    not_use_onnxsim=True
)


Automatic generation of each OP name started ========================================
Automatic generation of each OP name complete!

Model loaded ========================================================================

Model conversion started ============================================================
INFO: input_op_name: input shape: [1, 3, 256, 256] dtype: float32

INFO: 2 / 123
INFO: onnx_op_type: Conv onnx_op_name: node_Conv_505
INFO:  input_name.1: input shape: [1, 3, 256, 256] dtype: float32
INFO:  input_name.2: features.0.0.weight shape: [16, 3, 3, 3] dtype: float32
INFO:  input_name.3: features.0.0.weight_bias shape: [16] dtype: float32
INFO:  output_name.1: getitem shape: [1, 16, 128, 128] dtype: float32
INFO: tf_op_type: convolution_v2
INFO:  input.1.input: name: tf.compat.v1.pad_5/Pad:0 shape: (1, 258, 258, 3) dtype: <dtype: 'float32'> 
INFO:  input.2.weights: shape: (3, 3, 3, 16) dtype: <dtype: 'float32'> 
INFO:  input.3.bias: shape: (16,) dtype: <dtype: 'float32'> 
IN

In [17]:
output_dir = Path("converted_model_retry")

tflite_files = list(output_dir.rglob("*.tflite"))

for file in tflite_files:
    print(file)

float32_files = [
    file for file in tflite_files
    if "float32" in file.name.lower()
]

if not float32_files:
    print("Không tìm thấy file TFLite Float32.")
else:
    interpreter = tf.lite.Interpreter(
        model_path=str(float32_files[0])
    )

    interpreter.allocate_tensors()

    input_info = interpreter.get_input_details()[0]
    output_info = interpreter.get_output_details()[0]

    test_input = np.zeros(
        input_info["shape"],
        dtype=input_info["dtype"]
    )

    interpreter.set_tensor(input_info["index"], test_input)
    interpreter.invoke()

    output = interpreter.get_tensor(output_info["index"])

    print("Input shape:", input_info["shape"])
    print("Output shape:", output.shape)
    print("TFLite inference OK!")

converted_model_retry\garbage_classifier_float16.tflite
converted_model_retry\garbage_classifier_float32.tflite
Input shape: [  1 256 256   3]
Output shape: (1, 10)
TFLite inference OK!


# test single image

In [19]:
from PIL import Image

# Paths
MODEL_PATH = r"converted_model_retry\garbage_classifier_float32.tflite"
TRAIN_DIR = Path(r"data\dataset\train")
IMAGE_PATH = r"data\dataset\test\plastic\plastic_1561.jpg"

# Get class names in the same alphabetical order as ImageFolder
class_names = sorted(
    folder.name for folder in TRAIN_DIR.iterdir()
    if folder.is_dir()
)

# Load and preprocess the real image
image = Image.open(IMAGE_PATH).convert("RGB")
image = image.resize((256, 256), Image.Resampling.BILINEAR)

x = np.asarray(image, dtype=np.float32) / 255.0
x = np.expand_dims(x, axis=0)  # [1, 256, 256, 3]

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

x = x.astype(input_info["dtype"])

interpreter.set_tensor(input_info["index"], x)
interpreter.invoke()

output = interpreter.get_tensor(output_info["index"])[0]

# Convert logits to probabilities
exp_output = np.exp(output - np.max(output))
probabilities = exp_output / exp_output.sum()

pred_idx = int(np.argmax(probabilities))

print("Actual class:", Path(IMAGE_PATH).parent.name)
print("Predicted class:", class_names[pred_idx])
print(f"Confidence: {probabilities[pred_idx] * 100:.2f}%")

Actual class: plastic
Predicted class: plastic
Confidence: 99.91%


# test dataset

In [20]:
TEST_DIR = Path(r"data\dataset\test")

# Get class names
class_names = sorted(
    folder.name for folder in TEST_DIR.iterdir()
    if folder.is_dir()
)

# Load TFLite model
interpreter = tf.lite.Interpreter(
    model_path=MODEL_PATH
)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

correct = 0
total = 0

class_correct = np.zeros(len(class_names), dtype=int)
class_total = np.zeros(len(class_names), dtype=int)

confusion = np.zeros(
    (len(class_names), len(class_names)),
    dtype=int
)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Test all images
for true_idx, class_name in enumerate(class_names):

    class_dir = TEST_DIR / class_name

    for image_path in class_dir.rglob("*"):

        if image_path.suffix.lower() not in image_extensions:
            continue

        # Preprocess image
        image = Image.open(image_path).convert("RGB")
        image = image.resize(
            (256, 256),
            Image.Resampling.BILINEAR
        )

        x = np.asarray(image, dtype=np.float32) / 255.0
        x = np.expand_dims(x, axis=0)
        x = x.astype(input_info["dtype"])

        # Inference
        interpreter.set_tensor(input_info["index"], x)
        interpreter.invoke()

        output = interpreter.get_tensor(
            output_info["index"]
        )[0]

        pred_idx = int(np.argmax(output))

        # Calculate metrics
        total += 1
        class_total[true_idx] += 1
        confusion[true_idx, pred_idx] += 1

        if pred_idx == true_idx:
            correct += 1
            class_correct[true_idx] += 1

        if total % 100 == 0:
            print(f"Processed {total} images...")

# Results
print(f"\nTest Accuracy: {correct / total * 100:.2f}%")
print(f"Correct: {correct}/{total}")

print("\nAccuracy by class:")
for i, name in enumerate(class_names):
    if class_total[i] > 0:
        acc = class_correct[i] / class_total[i] * 100
        print(f"{name}: {acc:.2f}%")

print("\nConfusion Matrix (rows = actual, columns = predicted):")
print("Classes:", class_names)
print(confusion)

Processed 100 images...
Processed 200 images...
Processed 300 images...
Processed 400 images...
Processed 500 images...
Processed 600 images...
Processed 700 images...
Processed 800 images...
Processed 900 images...
Processed 1000 images...
Processed 1100 images...
Processed 1200 images...

Test Accuracy: 93.17%
Correct: 1146/1230

Accuracy by class:
battery: 90.79%
biological: 95.71%
cardboard: 92.96%
clothes: 98.42%
glass: 88.51%
metal: 95.70%
paper: 94.03%
plastic: 90.62%
shoes: 95.86%
trash: 82.61%

Confusion Matrix (rows = actual, columns = predicted):
Classes: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']
[[ 69   0   2   0   2   3   0   0   0   0]
 [  0  67   0   0   0   2   0   0   0   1]
 [  1   0 132   0   2   0   3   3   1   0]
 [  0   1   0 187   0   0   1   0   1   0]
 [  0   1   2   0 154   6   1   9   0   1]
 [  0   0   1   0   1  89   0   1   0   1]
 [  0   0   4   0   2   0 126   0   0   2]
 [  0   2   0   0   